> **Course notebook 2 of 5** | [Course home](../README.md) | [Previous: Tokenizer](01_tokenizer_from_scratch.ipynb) | [Next: Token Embeddings](03_token_embeddings_demo.ipynb)
>
> **Prerequisite:** Notebook 1; no neural-network knowledge is required. **Milestone:** explain how reusable subword pieces are learned and how one token-ID stream becomes next-token training windows.

# Byte Pair Encoding from Scratch

Tokenizers turn text into units a model can process. Three broad approaches are:

- **Word-based:** one token for each known word.
- **Character-based:** one token for each character.
- **Subword-based:** common text pieces become tokens, while rarer text is split into smaller pieces.

Byte Pair Encoding (BPE) is a subword method. It is historically important for GPT-style models because it offers a useful compromise between word and character tokenization. This notebook first builds the intuition manually and then uses GPT-2's practical byte-level BPE through `tiktoken`.

## Start here: why use subwords?

A word-level tokenizer needs a vocabulary entry for every known word. New words become unknown, and a very large vocabulary uses more memory. A character tokenizer avoids unknown words but turns ordinary text into long sequences.

**Byte Pair Encoding (BPE)** offers a middle ground. It begins with small pieces and repeatedly merges frequent neighboring pairs. Common patterns become larger tokens, while rare words can still be assembled from smaller pieces—like joining reusable letter magnets.

Keep two activities separate:

- **Training the tokenizer:** learn merge rules from a corpus.
- **Encoding text:** apply the learned rules to new text.

We first build the learning algorithm by hand. Then we use GPT-2's production tokenizer and turn its token IDs into input-target windows for next-token training.

## 1. Why BPE?

### Word-level tokenization

At word level, every word is a separate token. This usually creates short sequences, but a useful vocabulary can become very large. Words absent from the vocabulary create an **out-of-vocabulary (OOV)** problem. Related forms such as `boy` and `boys` also occupy separate vocabulary entries.

In [1]:
training_sentence = "My hobby is playing cricket"
word_tokens = training_sentence.split()
word_vocabulary = set(word_tokens)

test_sentence = "My hobby is playing football"
test_word_tokens = test_sentence.split()
tokens_with_unknown = [
    token if token in word_vocabulary else "<UNK>"
    for token in test_word_tokens
]

print("Training tokens:", word_tokens)
print("Tiny vocabulary:", sorted(word_vocabulary))
print("New sentence:   ", test_word_tokens)
print("Vocabulary view:", tokens_with_unknown)
print("'boy' and 'boys' would also be two separate entries.")

Training tokens: ['My', 'hobby', 'is', 'playing', 'cricket']
Tiny vocabulary: ['My', 'cricket', 'hobby', 'is', 'playing']
New sentence:    ['My', 'hobby', 'is', 'playing', 'football']
Vocabulary view: ['My', 'hobby', 'is', 'playing', '<UNK>']
'boy' and 'boys' would also be two separate entries.


## 2. Word vs Character Tokenization

A character tokenizer has a very small vocabulary and greatly reduces unknown-word problems. Its cost is much longer sequences, and useful larger language units are broken apart.

In [2]:
word = "dinosaur"
word_level = [word]
character_level = list(word)

print("Word tokens:     ", word_level)
print("Character tokens:", character_level)
print("Word tokenizer:     ", len(word_level), "token")
print("Character tokenizer:", len(character_level), "tokens")

Word tokens:      ['dinosaur']
Character tokens: ['d', 'i', 'n', 'o', 's', 'a', 'u', 'r']
Word tokenizer:      1 token
Character tokenizer: 8 tokens


## 3. Subword Tokenization

Subword tokenization keeps frequent pieces large and splits rarer words into reusable smaller pieces. Conceptually, we might see:

```text
boy  -> boy
boys -> boy + s

tokenization  -> token + ization
modernization -> modern + ization
```

These are only intuitive examples. Actual BPE does **not** manually choose linguistic roots or suffixes. Its merges are learned from frequency statistics.

## 4. BPE Merge Algorithm

We will train a tiny educational BPE model on four words with frequencies. Each word initially becomes characters followed by an explicit end-of-word marker, `</w>`. For example, `old` becomes `o l d </w>`.

In [3]:
word_frequencies = {
    "old": 7,
    "older": 3,
    "finest": 9,
    "lowest": 4,
}

def make_initial_corpus(word_frequencies):
    """Split every word into characters and append </w>."""
    corpus = {}
    for word, frequency in word_frequencies.items():
        symbols = tuple(list(word) + ["</w>"])
        corpus[symbols] = frequency
    return corpus

def print_corpus(corpus):
    for symbols, frequency in corpus.items():
        print(f"{' '.join(symbols):<28} frequency={frequency}")

bpe_corpus = make_initial_corpus(word_frequencies)

print("Exact tokens used for each word:")
for word, frequency in word_frequencies.items():
    word_tokens = list(word) + ["</w>"]
    print(f"{word!r:<10} -> {word_tokens}  (frequency={frequency})")

print("\nCompact corpus representation:")
print_corpus(bpe_corpus)

Exact tokens used for each word:
'old'      -> ['o', 'l', 'd', '</w>']  (frequency=7)
'older'    -> ['o', 'l', 'd', 'e', 'r', '</w>']  (frequency=3)
'finest'   -> ['f', 'i', 'n', 'e', 's', 't', '</w>']  (frequency=9)
'lowest'   -> ['l', 'o', 'w', 'e', 's', 't', '</w>']  (frequency=4)

Compact corpus representation:
o l d </w>                   frequency=7
o l d e r </w>               frequency=3
f i n e s t </w>             frequency=9
l o w e s t </w>             frequency=4


### Initial vocabulary

The initial symbols are simply the unique characters plus the end-of-word marker.

In [4]:
initial_symbols = sorted({
    symbol
    for symbols in bpe_corpus
    for symbol in symbols
})

print("Initial symbols:", initial_symbols)
print("Initial vocabulary size:", len(initial_symbols))

Initial symbols: ['</w>', 'd', 'e', 'f', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
Initial vocabulary size: 12


### Count adjacent pairs and merge one pair

Pair counts are weighted by word frequency. If `old` occurs 7 times, each adjacent pair inside `old` contributes 7 to its count.

In [5]:
from collections import defaultdict

def get_pair_frequencies(corpus):
    """Count adjacent symbol pairs, weighted by word frequency."""
    pair_frequencies = defaultdict(int)

    for symbols, word_frequency in corpus.items():
        for index in range(len(symbols) - 1):
            pair = (symbols[index], symbols[index + 1])
            pair_frequencies[pair] += word_frequency

    return dict(pair_frequencies)

def merge_pair(pair_to_merge, corpus):
    """Replace every occurrence of one adjacent pair with one symbol."""
    merged_corpus = {}

    for symbols, frequency in corpus.items():
        new_symbols = []
        index = 0

        while index < len(symbols):
            pair_matches = (
                index < len(symbols) - 1
                and (symbols[index], symbols[index + 1]) == pair_to_merge
            )

            if pair_matches:
                new_symbols.append(symbols[index] + symbols[index + 1])
                index += 2
            else:
                new_symbols.append(symbols[index])
                index += 1

        new_symbols = tuple(new_symbols)
        merged_corpus[new_symbols] = merged_corpus.get(new_symbols, 0) + frequency

    return merged_corpus

## 5. Manual BPE Example

At each iteration we count all adjacent pairs, choose the most frequent pair, merge it everywhere, and print the updated corpus. Ties are resolved by the order in which pairs were encountered, which is sufficient for this small demonstration.

In [6]:
num_merges = 10
bpe_corpus = make_initial_corpus(word_frequencies)
merge_rules = []

for iteration in range(1, num_merges + 1):
    pair_frequencies = get_pair_frequencies(bpe_corpus)

    if not pair_frequencies:
        print("No adjacent pairs remain.")
        break

    most_frequent_pair = max(pair_frequencies, key=pair_frequencies.get)
    frequency = pair_frequencies[most_frequent_pair]
    merge_rules.append(most_frequent_pair)

    print(f"Iteration {iteration}")
    print("Most frequent pair:", most_frequent_pair)
    print("Frequency:", frequency)

    bpe_corpus = merge_pair(most_frequent_pair, bpe_corpus)
    print("After merge:")
    print_corpus(bpe_corpus)
    print()

print("Learned merge order:")
for number, pair in enumerate(merge_rules, start=1):
    print(f"{number:>2}. {pair}")

Iteration 1
Most frequent pair: ('e', 's')
Frequency: 13
After merge:
o l d </w>                   frequency=7
o l d e r </w>               frequency=3
f i n es t </w>              frequency=9
l o w es t </w>              frequency=4

Iteration 2
Most frequent pair: ('es', 't')
Frequency: 13
After merge:
o l d </w>                   frequency=7
o l d e r </w>               frequency=3
f i n est </w>               frequency=9
l o w est </w>               frequency=4

Iteration 3
Most frequent pair: ('est', '</w>')
Frequency: 13
After merge:
o l d </w>                   frequency=7
o l d e r </w>               frequency=3
f i n est</w>                frequency=9
l o w est</w>                frequency=4

Iteration 4
Most frequent pair: ('o', 'l')
Frequency: 10
After merge:
ol d </w>                    frequency=7
ol d e r </w>                frequency=3
f i n est</w>                frequency=9
l o w est</w>                frequency=4

Iteration 5
Most frequent pair: ('ol', 'd')
Frequency:

### What BPE is actually learning

BPE begins with very small units and repeatedly merges frequently occurring neighboring units:

```text
characters
    -> frequent pairs
        -> larger subwords
            -> possibly common whole words
```

Common patterns tend to become larger tokens, while uncommon words may remain split into several pieces. **BPE does not inherently understand linguistic morphology.** If a root-like or suffix-like sequence becomes a token, it is because that sequence occurred frequently enough in the training data.

### Stopping criterion

Real BPE training does not need to continue until no pair remains. Training normally stops at a desired vocabulary size or after a maximum number of merges. Above, `num_merges = 10` is our deliberately simple stopping rule.

### Vocabulary-size tradeoff

```text
Small vocabulary              Large vocabulary
----------------              ----------------
more splitting                larger pieces
longer sequences              shorter sequences
smaller model matrices        larger embedding/output matrices
```

Vocabulary size is therefore a tradeoff between sequence length and the size of the model's vocabulary-dependent matrices.

## From the toy algorithm to a real tokenizer

Our example shows the core merge idea but omits many engineering details. GPT-2 uses byte-level BPE, which can represent arbitrary text without a normal unknown-word token. We will inspect it with `tiktoken`.

## 6. GPT-2 Tokenization with `tiktoken`

The manual code above teaches the merge mechanism. For practical GPT-2 tokenization, we will use OpenAI's `tiktoken` package. If it is not installed in your notebook environment, uncomment and run the installation line once.

In [7]:
# %pip install tiktoken

In [8]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

print("Tokenizer name:", tokenizer.name)
print("Vocabulary size:", tokenizer.n_vocab)
print("GPT-2 uses about 50k token IDs.")

Tokenizer name: gpt2
Vocabulary size: 50257
GPT-2 uses about 50k token IDs.


### Encode and decode normal text

In [9]:
text = "Hello, do you like tea?"
ids = tokenizer.encode(text)
decoded_text = tokenizer.decode(ids)

print("Original text:", text)
print("Token IDs:   ", ids)
print("Decoded text:", decoded_text)
print("Exact round trip:", decoded_text == text)

Original text: Hello, do you like tea?
Token IDs:    [15496, 11, 466, 345, 588, 8887, 30]
Decoded text: Hello, do you like tea?
Exact round trip: True


### Inspect individual token pieces

`decode_single_token_bytes` exposes the bytes represented by one token. We decode those bytes for display. A replacement symbol may appear when one token contains only part of a multi-byte Unicode character; decoding the complete ID sequence still recovers the original text.

In [10]:
def token_piece(tokenizer, token_id):
    token_bytes = tokenizer.decode_single_token_bytes(token_id)
    return token_bytes.decode("utf-8", errors="replace")

def print_token_pieces(tokenizer, token_ids):
    for token_id in token_ids:
        piece = token_piece(tokenizer, token_id)
        print(f"{token_id:<6} -> {piece!r}")

print_token_pieces(tokenizer, ids)

15496  -> 'Hello'
11     -> ','
466    -> ' do'
345    -> ' you'
588    -> ' like'
8887   -> ' tea'
30     -> '?'


## 7. Unknown Words

An invented word is unlikely to be one learned token, but GPT-2's byte-level BPE can represent it with smaller available pieces instead of a normal `<UNK>` token.

In [11]:
invented_word = "akwirwier"
invented_ids = tokenizer.encode(invented_word)

print("Original text:", invented_word)
print("Token IDs:", invented_ids)
print("Individual pieces:")
print_token_pieces(tokenizer, invented_ids)
print("Decoded text:", tokenizer.decode(invented_ids))
print("No <UNK> token was needed.")

Original text: akwirwier
Token IDs: [461, 86, 343, 86, 959]
Individual pieces:
461    -> 'ak'
86     -> 'w'
343    -> 'ir'
86     -> 'w'
959    -> 'ier'
Decoded text: akwirwier
No <UNK> token was needed.


### Word-level and BPE comparison

In [12]:
comparison_word = "unseenword"
word_level_result = "<UNK>"
bpe_ids = tokenizer.encode(comparison_word)
bpe_pieces = [token_piece(tokenizer, token_id) for token_id in bpe_ids]

print("Word-level tokenizer:")
print(comparison_word, "->", word_level_result)
print("\nGPT-2 BPE:")
print(comparison_word, "->", bpe_pieces)
print("BPE reuses smaller pieces and preserves the text.")

Word-level tokenizer:
unseenword -> <UNK>

GPT-2 BPE:
unseenword -> ['un', 'seen', 'word']
BPE reuses smaller pieces and preserves the text.


## 8. Special Tokens

GPT-2 uses `<|endoftext|>` to mark a document boundary. `tiktoken` treats special tokens carefully, so we explicitly permit this token with `allowed_special` when encoding it. GPT-2's end-of-text token is ID **50256**, and its vocabulary contains **50,257 IDs**, numbered 0 through 50256.

In [13]:
two_documents = """Hello, do you like tea?

<|endoftext|>

In the sunlit terraces of the palace..."""

document_ids = tokenizer.encode(
    two_documents,
    allowed_special={"<|endoftext|>"},
)

print("Token IDs:", document_ids)
print("<|endoftext|> ID:", tokenizer.eot_token)
print("Separator positions:", [
    index for index, token_id in enumerate(document_ids)
    if token_id == tokenizer.eot_token
])
print("Decoded text:")
print(tokenizer.decode(document_ids))

Token IDs: [15496, 11, 466, 345, 588, 8887, 30, 628, 50256, 198, 198, 818, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 986]
<|endoftext|> ID: 50256
Separator positions: [8]
Decoded text:
Hello, do you like tea?

<|endoftext|>

In the sunlit terraces of the palace...


### Important correction: characters versus bytes

Modern GPT-2-style BPE is specifically **byte-level BPE**. Its ultimate representation is not limited to ordinary English characters:

```text
Unicode text
    -> UTF-8 bytes
        -> byte-level representation
            -> BPE merges
```

This byte foundation lets the tokenizer represent arbitrary Unicode text without relying on a normal `<UNK>` token.

## 9. Edge Cases

These examples are for inspection, not benchmarking. Notice that punctuation, URLs, unusual letter sequences, digits, emoji, and Sinhala text can all round-trip through the tokenizer.

In [14]:
tests = [
    "hello",
    "unbelievable",
    "AI2026",
    "xqzvprk",
    "hello!!!",
    "email@example.com",
    "https://example.com",
    "\U0001F60A",  # smiling face emoji
    "\u0d86\u0dba\u0dd4\u0db6\u0ddd\u0dc0\u0db1\u0dca",  # Sinhala: ayubowan
]

for test_text in tests:
    test_ids = tokenizer.encode(test_text)
    test_pieces = [token_piece(tokenizer, token_id) for token_id in test_ids]

    print("Text:", test_text)
    print("Token IDs:", test_ids)
    print("Token pieces:", test_pieces)
    print("Number of tokens:", len(test_ids))
    print("Decoded text:", tokenizer.decode(test_ids))
    print("-" * 60)

Text: hello
Token IDs: [31373]
Token pieces: ['hello']
Number of tokens: 1
Decoded text: hello
------------------------------------------------------------
Text: unbelievable
Token IDs: [403, 6667, 11203, 540]
Token pieces: ['un', 'bel', 'iev', 'able']
Number of tokens: 4
Decoded text: unbelievable
------------------------------------------------------------
Text: AI2026
Token IDs: [20185, 1238, 2075]
Token pieces: ['AI', '20', '26']
Number of tokens: 3
Decoded text: AI2026
------------------------------------------------------------
Text: xqzvprk
Token IDs: [87, 80, 89, 85, 1050, 74]
Token pieces: ['x', 'q', 'z', 'v', 'pr', 'k']
Number of tokens: 6
Decoded text: xqzvprk
------------------------------------------------------------
Text: hello!!!
Token IDs: [31373, 10185]
Token pieces: ['hello', '!!!']
Number of tokens: 2
Decoded text: hello!!!
------------------------------------------------------------
Text: email@example.com
Token IDs: [12888, 31, 20688, 13, 785]
Token pieces: ['emai

## 10. Summary

```text
Word tokenizer
-> short sequences
-> huge vocabulary
-> OOV problem

Character/byte tokenizer
-> tiny vocabulary
-> handles arbitrary text
-> very long sequences

BPE
-> compromise
-> frequent sequences become larger tokens
-> rare strings split into smaller pieces
-> manageable vocabulary
-> no ordinary UNK requirement for byte-level BPE
```

The complete path forward is:

```text
Raw text
    -> BPE tokenizer
        -> subword / byte-derived tokens
            -> token IDs

NEXT LECTURE:
data sampling, context windows, and batches
    -> embeddings
        -> Transformer
```

The BPE lecture originally stopped here. The continuation below now implements data sampling, context windows, and batches before embeddings.

# From token IDs to training examples

A tokenizer produces one long stream of IDs. Language-model training needs shorter examples in which each target is the next token after its matching input. The remaining sections create those shifted windows and batch them with PyTorch.

In [15]:
# Reuse the tokenizer created in the GPT-2 tiktoken section above.
print("Tokenizer in use:", tokenizer.name)
assert tokenizer.name == "gpt2"

Tokenizer in use: gpt2


## 11. Load the text corpus

The notebook prefers `the-verdict.txt` when that file is beside the notebook. If it is unavailable, an original repeated fallback passage keeps every demonstration runnable, including the longer-context batch.

In [16]:
from pathlib import Path

corpus_path = Path("the-verdict.txt")

if corpus_path.exists():
    raw_text = corpus_path.read_text(encoding="utf-8")
    corpus_source = str(corpus_path)
else:
    fallback_passage = (
        "At the edge of the quiet town, Mira opened her notebook and recorded "
        "what she had observed that morning. The market was waking slowly; "
        "lamps faded as sunlight crossed the roofs, and distant carts moved "
        "along the road. She compared yesterday's notes with today's details, "
        "asked a careful question, and wrote a clear answer for the next reader."
    )
    raw_text = (fallback_passage + "\n\n") * 80
    corpus_source = "built-in fallback text"

print("Corpus source:", corpus_source)
print("Number of characters:", len(raw_text))
print("\nFirst 300 characters:")
print(raw_text[:300])

Corpus source: built-in fallback text
Number of characters: 27520

First 300 characters:
At the edge of the quiet town, Mira opened her notebook and recorded what she had observed that morning. The market was waking slowly; lamps faded as sunlight crossed the roofs, and distant carts moved along the road. She compared yesterday's notes with today's details, asked a careful question, and


## 12. Tokenize the complete text with GPT-2

`len(encoded_text)` is the number of **GPT-2 tokens in this text**. It is not the GPT-2 vocabulary size.

In [17]:
encoded_text = tokenizer.encode(raw_text)

print("Number of text tokens:", len(encoded_text))
print("First 20 token IDs:", encoded_text[:20])
print("Decoded first 20 tokens:", repr(tokenizer.decode(encoded_text[:20])))

Number of text tokens: 5679
First 20 token IDs: [2953, 262, 5743, 286, 262, 5897, 3240, 11, 7381, 64, 4721, 607, 20922, 290, 6264, 644, 673, 550, 6515, 326]
Decoded first 20 tokens: 'At the edge of the quiet town, Mira opened her notebook and recorded what she had observed that'


## 13. Create the lecture's encoded sample

We skip the first 50 tokens only to begin at a more interesting point in the passage, following the lecture demonstration. This is not a general preprocessing requirement.

In [18]:
encoded_sample = encoded_text[50:]

print("First 20 sample IDs:", encoded_sample[:20])
print("Decoded sample start:", repr(tokenizer.decode(encoded_sample[:20])))

First 20 sample IDs: [1909, 338, 3307, 11, 1965, 257, 8161, 1808, 11, 290, 2630, 257, 1598, 3280, 329, 262, 1306, 9173, 13, 198]
Decoded sample start: " today's details, asked a careful question, and wrote a clear answer for the next reader.\n"


## 14. Build one input-target pair manually

For next-token prediction, `Y` is `X` shifted by exactly one token. Conceptually, `[1, 2, 3, 4]` predicts `[2, 3, 4, 5]`.

In [19]:
context_size = 4
x = encoded_sample[:context_size]
y = encoded_sample[1:context_size + 1]

print("Input X: ", x)
print("Target Y:", y)
print("Decoded X:", repr(tokenizer.decode(x)))
print("Decoded Y:", repr(tokenizer.decode(y)))
print("Y equals X shifted by one token:", x[1:] == y[:-1])

Input X:  [1909, 338, 3307, 11]
Target Y: [338, 3307, 11, 1965]
Decoded X: " today's details,"
Decoded Y: "'s details, asked"
Y equals X shifted by one token: True


### Four individual next-token tasks

A sequence with `context_size = 4` supplies four target positions: predict token 1 from the first context, token 2 from the first two context tokens, and so on.

In [20]:
print("Token-ID view:")
for i in range(1, context_size + 1):
    context = x[:i]
    desired = y[i - 1]
    print(context, "---->", desired)

print("\nDecoded-text view:")
for i in range(1, context_size + 1):
    context = x[:i]
    desired = y[i - 1]
    print(
        repr(tokenizer.decode(context)),
        "---->",
        repr(tokenizer.decode([desired])),
    )

print("\nTarget positions in this sequence:", context_size)

Token-ID view:
[1909] ----> 338
[1909, 338] ----> 3307
[1909, 338, 3307] ----> 11
[1909, 338, 3307, 11] ----> 1965

Decoded-text view:
' today' ----> "'s"
" today's" ----> ' details'
" today's details" ----> ','
" today's details," ----> ' asked'

Target positions in this sequence: 4


## 15. Context length

```text
context length / max_length
=
maximum number of tokens contained in one input training sequence
```

The unit is **tokens**, not words. GPT-2 uses BPE, so one word can correspond to multiple tokens. `max_length = 4` means every input row contains four GPT-2 token IDs.

## 16. Sliding windows and stride

Context length is the window width. Stride is how far the input window moves before the next sample is created. The target shift is always one token; stride controls movement **between samples**, not the `X` to `Y` shift.

In [21]:
simple_tokens = list(range(1, 10))
manual_context_length = 4

for manual_stride in (1, 4):
    windows = [
        simple_tokens[start:start + manual_context_length]
        for start in range(0, len(simple_tokens) - manual_context_length + 1, manual_stride)
    ]
    print(f"stride = {manual_stride}")
    for number, window in enumerate(windows[:3], start=1):
        print(f"X{number} = {window}")
    print()

print("The Y sequence for every window is still shifted by exactly one token.")

stride = 1
X1 = [1, 2, 3, 4]
X2 = [2, 3, 4, 5]
X3 = [3, 4, 5, 6]

stride = 4
X1 = [1, 2, 3, 4]
X2 = [5, 6, 7, 8]

The Y sequence for every window is still shifted by exactly one token.


## Why use a Dataset and DataLoader?

A PyTorch `Dataset` stores individual input-target pairs. A `DataLoader` groups them into batches and can shuffle or load them efficiently. Keeping these jobs separate makes the same data easy to reuse with different batch settings.

## 17. PyTorch Dataset and DataLoader

We now turn the same sliding-window idea into objects PyTorch can use.

In [22]:
import torch
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0+cpu


### `GPTDatasetV1`

The constructor tokenizes the complete text once, creates sliding input/target chunks, and stores each chunk as a `torch.long` tensor.

In [23]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        if max_length < 1:
            raise ValueError("max_length must be at least 1")
        if stride < 1:
            raise ValueError("stride must be at least 1")

        token_ids = tokenizer.encode(txt)
        self.input_ids = []
        self.target_ids = []

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(
                torch.tensor(input_chunk, dtype=torch.long)
            )
            self.target_ids.append(
                torch.tensor(target_chunk, dtype=torch.long)
            )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

### What `__len__` and `__getitem__` mean

```text
__len__()
-> tells PyTorch how many training samples exist

__getitem__(idx)
-> returns one input-target pair
```

For example, `dataset[0]` returns an input `[t1, t2, t3, t4]` and target `[t2, t3, t4, t5]`.

In [24]:
dataset_example = GPTDatasetV1(raw_text, tokenizer, max_length=4, stride=1)
sample_input, sample_target = dataset_example[0]

print("Number of dataset samples:", len(dataset_example))
print("dataset[0] input: ", sample_input)
print("dataset[0] target:", sample_target)

Number of dataset samples: 5675
dataset[0] input:  tensor([2953,  262, 5743,  286])
dataset[0] target: tensor([ 262, 5743,  286,  262])


### Create the DataLoader helper

In [25]:
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    gpt2_tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, gpt2_tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader

### Dataset versus DataLoader

```text
Dataset
-> defines individual X/Y training samples

DataLoader
-> groups samples into batches
-> optionally shuffles them
-> handles data loading
-> provides batches during training
```

## 18. First DataLoader experiment: batch 1, context 4, stride 1

In [33]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False,
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs:\n", inputs)
print("Targets:\n", targets)
print("Input shape: ", inputs.shape)
print("Target shape:", targets.shape)
print("1 = batch size; 4 = context length")

Inputs:
 tensor([[2953,  262, 5743,  286]])
Targets:
 tensor([[ 262, 5743,  286,  262]])
Input shape:  torch.Size([1, 4])
Target shape: torch.Size([1, 4])
1 = batch size; 4 = context length


### Decode the first batch test
Decoding makes the one-token target shift visible in the original text.

In [34]:
first_input_ids = inputs[0].tolist()
first_target_ids = targets[0].tolist()

print("Input IDs:  ", first_input_ids)
print("Target IDs: ", first_target_ids)
print("Input text: ", repr(tokenizer.decode(first_input_ids)))
print("Target text:", repr(tokenizer.decode(first_target_ids)))

Input IDs:   [2953, 262, 5743, 286]
Target IDs:  [262, 5743, 286, 262]
Input text:  'At the edge of'
Target text: ' the edge of the'


### Fetch the second batch

With `stride = 1`, the second input window moves only one token, so the two batches overlap heavily.

In [35]:
inputs2, targets2 = next(data_iter)

print("Batch 1 input:", inputs[0].tolist())
print("Batch 2 input:", inputs2[0].tolist())
print("Batch 1 text: ", repr(tokenizer.decode(inputs[0].tolist())))
print("Batch 2 text: ", repr(tokenizer.decode(inputs2[0].tolist())))
print("Overlapping IDs:", inputs[0][1:].tolist() == inputs2[0][:-1].tolist())

Batch 1 input: [2953, 262, 5743, 286]
Batch 2 input: [262, 5743, 286, 262]
Batch 1 text:  'At the edge of'
Batch 2 text:  ' the edge of the'
Overlapping IDs: True


## 19. Compare stride 1 with stride 4

Stride 1 creates overlapping windows. Stride 4 creates non-overlapping input windows when the context length is also 4. Stride does not generally have to equal context length.

In [29]:
for compared_stride in (1, 4):
    compared_loader = create_dataloader_v1(
        raw_text,
        batch_size=1,
        max_length=4,
        stride=compared_stride,
        shuffle=False,
    )
    compared_iterator = iter(compared_loader)

    print(f"stride = {compared_stride}")
    for window_number in range(1, 4):
        window_inputs, _ = next(compared_iterator)
        window_ids = window_inputs[0].tolist()
        print(
            f"X{window_number}: {window_ids} -> "
            f"{tokenizer.decode(window_ids)!r}"
        )
    print()

stride = 1
X1: [2953, 262, 5743, 286] -> 'At the edge of'
X2: [262, 5743, 286, 262] -> ' the edge of the'
X3: [5743, 286, 262, 5897] -> ' edge of the quiet'

stride = 4
X1: [2953, 262, 5743, 286] -> 'At the edge of'
X2: [262, 5897, 3240, 11] -> ' the quiet town,'
X3: [7381, 64, 4721, 607] -> ' Mira opened her'



## 20. Demonstrate `batch_size = 8`

In [30]:
batch_dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False,
)
batch_inputs, batch_targets = next(iter(batch_dataloader))

print("Input shape: ", batch_inputs.shape)
print("Target shape:", batch_targets.shape)
print("8 sequences x 4 positions =", batch_inputs.numel(), "target positions")

Input shape:  torch.Size([8, 4])
Target shape: torch.Size([8, 4])
8 sequences x 4 positions = 32 target positions


### `batch_size`, `num_workers`, and `drop_last`

`batch_size` is the number of training sequences processed together before one optimizer training-step update. Here, 8 sequences times 4 token positions gives 32 target positions. It is not a CPU process count.

`num_workers` is the number of worker processes the DataLoader can use to fetch and prepare data. It controls data-loading parallelism, not training grouping. DataLoader workers do not independently modify model weights. `num_workers=0` is the most reliable default inside a notebook.

With `drop_last=True`, an incomplete final batch containing fewer samples than `batch_size` is discarded.

## 21. Optional longer-context experiment

This is only a larger demonstration of the same mechanism. These values are not being presented as the exact GPT-2 training hyperparameters.

In [31]:
long_dataloader = create_dataloader_v1(
    raw_text,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=False,
)

try:
    long_inputs, long_targets = next(iter(long_dataloader))
except StopIteration:
    long_inputs = long_targets = None
    print("The selected corpus is too short for a complete [4, 256] batch.")
else:
    print("Input shape: ", long_inputs.shape)
    print("Target shape:", long_targets.shape)
    print("4 x 256 =", long_inputs.numel(), "target token positions")

Input shape:  torch.Size([4, 256])
Target shape: torch.Size([4, 256])
4 x 256 = 1024 target token positions


## 22. Sanity checks

Every input and target batch must have matching shapes. Within each row, all but the first input token must equal all but the last target token because the target is shifted by one position.

In [32]:
assert inputs.shape == targets.shape == torch.Size([1, 4])
assert torch.equal(inputs[0][1:], targets[0][:-1])

assert batch_inputs.shape == batch_targets.shape == torch.Size([8, 4])
for row_index in range(batch_inputs.shape[0]):
    assert torch.equal(
        batch_inputs[row_index][1:],
        batch_targets[row_index][:-1],
    )
assert batch_inputs.shape[1] == 4

if long_inputs is not None:
    assert long_inputs.shape == long_targets.shape == torch.Size([4, 256])
    assert torch.equal(long_inputs[:, 1:], long_targets[:, :-1])

print("All shape and one-token alignment checks passed.")

All shape and one-token alignment checks passed.


## 23. Final visual summary

```text
Raw text
-> GPT-2 tiktoken
-> token IDs
-> sliding windows

X = [t1, t2, t3, t4]
Y = [t2, t3, t4, t5]

-> GPTDatasetV1
-> individual training samples
-> DataLoader
-> batches

X shape: [batch_size, context_length]
Y shape: [batch_size, context_length]

-> NEXT STAGE: token embeddings
```

| Parameter | Meaning |
|---|---|
| `max_length` / context length | Number of tokens in one training sequence |
| `stride` | How far the window moves to create the next sequence |
| `batch_size` | Number of sequences grouped together |
| `num_workers` | Parallel DataLoader workers |
| `shuffle` | Whether dataset sample order is randomized |
| `drop_last` | Whether the incomplete final batch is discarded |

Most importantly:

```text
X -> Y shift = always exactly 1 token
stride = movement between consecutive input samples
```

These are different quantities. The notebook stops with correctly aligned input and target batches ready for the next lecture on token/vector embeddings.

### Checkpoint

You should now be able to explain why subwords balance vocabulary size and sequence length, how BPE learns merges, and why training targets are the input stream shifted by one token.

**Next:** Notebook 3 turns token IDs into learnable vectors.